In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random as rd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

In [14]:
def mse_loss(Xb, y, w):
    y_hat = predict(Xb, w)
    return np.mean((y_hat - y) ** 2)


def mse_grad(Xb, y, w):
    n = len(y)

    return (2/n) * (Xb.T @ (Xb @ w - y))




def predict(Xb, w):
    y_hat = Xb @ w
    return y_hat

In [15]:
def add_bias_column(X):
    n = X.shape[0]

    ones = np.ones((n, 1), dtype=X.dtype)

    Xb = np.hstack([ones, X])

    return Xb

def main():
    df = pd.read_csv('Chevy_US_Data.csv')


    # For Classification

    df['Sales_Increase'] = (df['YOY Change'] > 0).astype(int) # Here we are predicting which sales increased or decreased


    print(df.head())


    SEED = rd.randrange(3320, 4320)
    
    
    
    TARGET_COL = "Total Sales"


    test_size = 0.15
    val_size = 0.15


    y = df[TARGET_COL].to_numpy(dtype=np.float64)



    X_df = df.drop(columns=[TARGET_COL])


    X_train, X_test, y_train, y_test = train_test_split( X_df, y, test_size=test_size, random_state=SEED)



    val_fraction = val_size / (1.0 - test_size)


    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=val_fraction, random_state=SEED)



    from pandas.api.types import is_numeric_dtype


    numeric_features = [c for c in X_df.columns if is_numeric_dtype(X_df[c])]

    categorical_features = [c for c in X_df.columns if c not in numeric_features]

    num_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])





    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])




    pre = ColumnTransformer(transformers=[
        ("num", num_pipe, numeric_features),
        ("cat", cat_pipe, categorical_features)
    ],
        remainder="drop",)



    full_pipeline = Pipeline(steps=[
        ("pre", pre),
        ("reg", Ridge(alpha=1.0))
    ])


    param_grid = {
        'reg__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
    }

    grid = GridSearchCV(full_pipeline, param_grid, scoring='neg_mean_squared_error', cv=5)


    grid.fit(X_train, y_train)


    print(grid.best_params_)


    full_pipeline.set_params(reg__alpha=grid.best_params_['reg__alpha'])


    full_pipeline.fit(X_train, y_train)


    y_train_p = full_pipeline.predict(X_train)


    print("Train MSE: ", mean_squared_error(y_train, y_train_p))


    y_val_p = full_pipeline.predict(X_val)



    print("Validation MSE: ", mean_squared_error(y_val, y_val_p))

    






main()





   Year     Jan     Feb     Mar     Apr     May     Jun     Jul     Aug  \
0  2005  168143  176662  249987  220997  226318  316445  294630  191875   
1  2006  168585  170775  210113  202877  202278  234890  237036  203348   
2  2007  144439  183362  207024  182625  218452  181673  180406  223412   
3  2008  146338  162185  160127  152041  162282  154905  137188  186023   
4  2009   77307   75878   94850  116046  129016  106982  125332  168669   

      Sep     Oct     Nov     Dec  Total Sales  YOY Change  US Marketshare  \
0  197959  145032  159817  236644      2584509      8.9383          0.1510   
1  186073  167655  165836  194969      2344435     -0.0929          0.1406   
2  189013  173717  148015  187200      2219338     -0.0534          0.1368   
3  173890  108107   95732  137650      1776468     -0.1996          0.1332   
4  102408  117306  100223  135583      1349600     -0.2403          0.1284   

   Marketshare change Best Selling Model Second Best Selling Model  \
0         